# Simulate RI Logit

## Packages

In [1]:
import numpy as np

from scipy.optimize import minimize
from src.simulation.simulate_logit import SimulateRILogit
from src.model.logit import RILogit

## Global Parameters

#TODO find a simulation that gives rise to marg all interiors and then estimate the logit MLE. 

In [2]:
J = 5 # Number of products
N = 3 # Number of states per products
llambda = 0.1 # Information Cost
I = 1 # Number of Individuals
n_sim = 100000 # Number of simulation for each individuals
state = (0, 1, 0, 0, 1)

## Individuals Priors

In [41]:
list_ppi = [
    np.array([np.random.dirichlet(np.ones(N)) for _ in range(J)]).T for _ in range(I)
]

In [42]:
list_ppi

[array([[0.19472821, 0.00282684, 0.14576522, 0.31450615, 0.23489874],
        [0.0811083 , 0.72000998, 0.6444873 , 0.36073916, 0.36927968],
        [0.72416349, 0.27716318, 0.20974749, 0.32475469, 0.39582158]])]

## Simulate Individuals Characteristics

for now only one individual

In [43]:
mean_i = np.repeat([1], repeats=I).reshape(I, 1)
X_i = np.random.normal(mean_i, scale=0.05)

In [44]:
X_i

array([[0.84205244]])

## Simulate Products Characteristics

In [45]:
mean_j = np.repeat([1.2], repeats=J).reshape(J, 1)
X_j = np.random.normal(mean_j, scale=1)

In [8]:
X_j

array([[2.83970055],
       [1.91274411],
       [2.9989002 ]])

## Simulate States Characteristics

In [46]:
mean_omega = np.repeat([1.5], repeats=N).reshape(N, 1)
X_omega = np.random.normal(mean_omega, scale=1)

In [10]:
X_omega

array([[0.47774161],
       [2.51675794],
       [2.48014276]])

## Define True Parameters

In [47]:
theta_i = np.ones([1])
theta_j = np.array([1])
theta_w = np.array([1])

## Obtain Utility from Characteristics

To assess the performance of our estimators we have to get payoff and prior matrix as function of product characteristics $X_j$ individual characteristics $X_i$ and preferences $\theta \in \Theta$ to be estimated.

$u_{ij\omega} = \theta_1 X_i + \theta_2 X_j + \theta_3 X_\omega $ (additively separable) 

$u_{ij\omega} = h(\theta, X_i, X_j, X_\omega) $

If we want to model heterogeneity, we would compute utilities from different $\theta$.

$\theta$ is of shape $(I, J, N)$

In [48]:
u_mat = (
    # np.einsum("ic,c->i", X_i, theta_j)[:, None, None]
    + np.einsum("jc,c->j", X_j, theta_j)[None, None, :]
    + np.einsum("wc,c->w", X_omega, theta_w)[None, :, None]
)

list_u_mat = [u_mat[i] for i in range(I)]

In [49]:
u_mat

array([[[5.1180767 , 6.79002419, 5.30993611, 3.34815508, 4.94252717],
        [3.31920247, 4.99114996, 3.51106187, 1.54928085, 3.14365294],
        [3.48559402, 5.15754151, 3.67745343, 1.7156724 , 3.31004449]]])

In [14]:
list_u_mat[0].shape

(3, 3)

## Simulate Logit Choice

Using Blahut–Arimoto solver

In [52]:
list_classes_BA = [
    SimulateRILogit(u_mat=u_mat, ppi=ppi, llambda=llambda)
    for u_mat, ppi in zip(list_u_mat, list_ppi)
]
list_indexes_BA = [cl.get_states().index(state) for cl in list_classes_BA]
list_choices_BA = [cl.simulate(n_sim=n_sim, states=state) for cl in list_classes_BA]
list_dist_BA = [cl.get_logit()[index] for cl, index in zip(list_classes_BA, list_indexes_BA)]
list_marg_BA = [cl.get_marg() for cl in list_classes_BA]

Blahut–Arimoto Solver:   0%|          | 10/10000 [00:00<00:04, 2413.71iter/s, Error=[3.5062982e-13]]


In [53]:
list_marg_BA

[array([[1.30828971e-04],
        [9.15262331e-01],
        [8.46068399e-02],
        [0.00000000e+00],
        [0.00000000e+00]])]

In [36]:
list_classes_SQP = [
    SimulateRILogit(u_mat=u_mat, ppi=ppi, llambda=llambda, method="SQP")
    for u_mat, ppi in zip(list_u_mat, list_ppi)
]
list_indexes_SQP = [cl.get_states().index(state) for cl in list_classes_SQP]
list_choices_SQP = [cl.simulate(n_sim=n_sim, states=state) for cl in list_classes_SQP]
list_dist_SQP = [cl.get_logit()[index] for cl, index in zip(list_classes_SQP, list_indexes_SQP)]

SQP Solver:   0%|          | 0/10000 [00:00<?, ?iter/s]


Warning algorithm did not converge


SQP Solver:   0%|          | 1/10000 [00:00<00:40, 246.61iter/s, Error=0]


In [54]:
u_mat

array([[[5.1180767 , 6.79002419, 5.30993611, 3.34815508, 4.94252717],
        [3.31920247, 4.99114996, 3.51106187, 1.54928085, 3.14365294],
        [3.48559402, 5.15754151, 3.67745343, 1.7156724 , 3.31004449]]])

## Estimation

In [55]:
n_j = list_choices_BA[0][1] * n_sim

In [56]:
n_j

array([[1.1000e+01, 3.0722e+04, 6.9267e+04, 0.0000e+00, 0.0000e+00]])

In [58]:
theta_init = np.array([0.5, 0.8])

In [61]:
u_mat = (
    + np.einsum("jc,c->j", X_j, theta_i)[None, :]
    + np.einsum("wc,c->w", X_omega, theta_j)[:, None]
)
u_vec = u_mat[state, np.arange(len(state))]

In [63]:
u_vec

array([5.1180767 , 4.99114996, 5.30993611, 3.34815508, 3.14365294])

In [72]:
marg

array([[1.30828971e-04],
       [9.15262331e-01],
       [8.46068399e-02],
       [0.00000000e+00],
       [0.00000000e+00]])

treat case with marg = zero

In [76]:
marg = list_marg_BA[0].reshape(1,-1)

def log_likelihood(params: np.array):

    u_mat = (
                + np.einsum("jc,c->j", X_j, params[:j_dim])[None, :]
                + np.einsum("wc,c->w", X_omega, params[j_dim:w_dim])[:, None]
            )
    u_vec = u_mat[state, np.arange(len(state))]
    b_vec = np.exp(u_vec / llambda)
    print(b_vec)
    print(marg * b_vec)
    denominator = np.sum(marg * b_vec)

    return - np.sum(n_j * (np.log(marg) + np.log(b_vec) - np.log(denominator)))

result = minimize(
    log_likelihood,
    theta_init,
    options={"disp": True},
)

theta_new = result.x

[7.67745737e+03 3.27948201e+07 2.00370979e+04 1.10123866e+00
 3.19165877e+03]
[[1.00443384e+00 3.00158635e+07 1.69527554e+03 0.00000000e+00
  0.00000000e+00]]
[7.67745942e+03 3.27948371e+07 2.00371038e+04 1.10123866e+00
 3.19165954e+03]
[[1.00443411e+00 3.00158790e+07 1.69527604e+03 0.00000000e+00
  0.00000000e+00]]
[7.67745737e+03 3.27948201e+07 2.00370979e+04 1.10123866e+00
 3.19165877e+03]
[[1.00443384e+00 3.00158635e+07 1.69527554e+03 0.00000000e+00
  0.00000000e+00]]
         Current function value: nan
         Iterations: 0
         Function evaluations: 3
         Gradient evaluations: 1


/var/folders/7l/_s01npq1225c41_90vkwthlc0000gn/T/ipykernel_6164/3420635038.py:15: RuntimeWarning: divide by zero encountered in log
  return - np.sum(n_j * (np.log(marg) + np.log(b_vec) - np.log(denominator)))
/var/folders/7l/_s01npq1225c41_90vkwthlc0000gn/T/ipykernel_6164/3420635038.py:15: RuntimeWarning: invalid value encountered in multiply
  return - np.sum(n_j * (np.log(marg) + np.log(b_vec) - np.log(denominator)))


In [69]:
theta_new

array([0.5, 0.8])

In [133]:
u_mat = (
    np.einsum("ic,c->i", X_i, theta_i)[:, None, None]
    + np.einsum("jc,c->j", X_j, theta_j)[None, None, :]
    + np.einsum("wc,c->w", X_omega, theta_w)[None, :, None]
)

In [31]:
n_j = list_choices_BA[0][1] * n_sim 

j_dim = X_j.shape[1]
w_dim = X_omega.shape[1]

C = j_dim + w_dim

theta_init = np.array([0.5, 0.8])

ppi = list_ppi[0]


def EM(theta_init: np.array, state: tuple, n_iter, cvg_criterion):

    i = 0
    error = np.inf

    theta = np.copy(theta_init)

    while error > cvg_criterion and i < n_iter:

        u_mat = (
            + np.einsum("jc,c->j", X_j, theta[:j_dim])[None, :]
            + np.einsum("wc,c->w", X_omega, theta[j_dim:w_dim])[:, None]
        )

        RI = RILogit(u_mat=u_mat, ppi=ppi, llambda=llambda)

        marg = RI.get_marg()


        def log_likelihood(params: np.array):

            u_mat = (
                +np.einsum("jc,c->j", X_j, params[:j_dim])[None, :]
                + np.einsum("wc,c->w", X_omega, params[j_dim:w_dim])[:, None]
            )
            u_vec = u_mat[np.arange(len(state)), state]
            b_vec = np.exp(u_vec / llambda)

            denominator = np.sum(marg * b_vec)

            return - np.sum(n_j * (np.log(marg) + np.log(b_vec) - np.log(denominator)))

        result = minimize(
            log_likelihood,
            theta,
            options={"disp": True},
        )

        theta_new = result.x

        error = np.sum((theta_new - theta)**2)

        i += 1


    return theta, marg, error, i


In [32]:
EM(theta_init, state, 1000, 1e-4)

Blahut–Arimoto Solver:   0%|          | 1/10000 [00:00<00:28, 355.09iter/s, Error=[0.]]

         Current function value: inf
         Iterations: 0
         Function evaluations: 3
         Gradient evaluations: 1



/var/folders/7l/_s01npq1225c41_90vkwthlc0000gn/T/ipykernel_6164/1032692309.py:43: RuntimeWarning: divide by zero encountered in log
  return - np.sum(n_j * (np.log(marg) + np.log(b_vec) - np.log(denominator)))


(array([0.5, 0.8]),
 array([[0.],
        [0.],
        [1.]]),
 np.float64(0.0),
 1)